In [1]:
# Read Bronze ASQP table

asqp_df = spark.read.table("bronze_asqp_airline_operations")

print("Row Count:", asqp_df.count())

asqp_df.printSchema()

display(asqp_df.limit(10))

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 3, Finished, Available, Finished, False)

Row Count: 10590
root
 |-- Date: date (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Actual_departures: long (nullable = true)
 |-- Actual_Arrivals: long (nullable = true)
 |-- Departure_Cancellations: long (nullable = true)
 |-- Arrival_Cancellations: long (nullable = true)
 |-- Departure_Diversions: long (nullable = true)
 |-- Arrival_Diversions: long (nullable = true)
 |-- On-Time_Arrivals: long (nullable = true)
 |-- % On-Time_Gate_Departures: decimal(34,6) (nullable = true)
 |-- % On-Time_Gate_Arrivals: decimal(34,6) (nullable = true)
 |-- Average_Gate_Departures_Delay: long (nullable = true)
 |-- Average__Gate_Arrival_delay: long (nullable = true)
 |-- Average_Block_Delay: long (nullable = true)
 |-- Average_Taxi_out_time: long (nullable = true)
 |-- Average_Taxi_in_Time: long (nullable = true)
 |-- Delayed_Arrivals: long (nullable = true)
 |-- Average_Delay_Per_Delayed_Arrival: long (nullable = true)



SynapseWidget(Synapse.DataFrame, 1b82c03d-7629-4194-be25-91e8ff31e4b4)

In [2]:
from pyspark.sql.functions import col, count, when

display(
    asqp_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in asqp_df.columns
    ])
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5db585c5-3adb-4c2d-a2ae-9e7c95e83811)

In [3]:
print("Total Rows :", asqp_df.count())

print("Distinct Rows :", asqp_df.distinct().count())

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 5, Finished, Available, Finished, False)

Total Rows : 10590
Distinct Rows : 10590


In [4]:
from pyspark.sql.functions import min, max, countDistinct

asqp_df.select(
    min("Date").alias("MinDate"),
    max("Date").alias("MaxDate"),
    countDistinct("Date").alias("DistinctDates")
).show()

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 6, Finished, Available, Finished, False)

+----------+----------+-------------+
|   MinDate|   MaxDate|DistinctDates|
+----------+----------+-------------+
|2024-01-01|2024-01-31|           31|
+----------+----------+-------------+



In [5]:
from pyspark.sql.functions import count

(
    asqp_df
    .groupBy(asqp_df.columns)
    .agg(count("*").alias("duplicate_count"))
    .filter("duplicate_count > 1")
    .show(20, False)
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 7, Finished, Available, Finished, False)

+----+------------+-----------------+---------------+-----------------------+---------------------+--------------------+------------------+----------------+-------------------------+-----------------------+-----------------------------+---------------------------+-------------------+---------------------+--------------------+----------------+---------------------------------+---------------+
|Date|Airport_Code|Actual_departures|Actual_Arrivals|Departure_Cancellations|Arrival_Cancellations|Departure_Diversions|Arrival_Diversions|On-Time_Arrivals|% On-Time_Gate_Departures|% On-Time_Gate_Arrivals|Average_Gate_Departures_Delay|Average__Gate_Arrival_delay|Average_Block_Delay|Average_Taxi_out_time|Average_Taxi_in_Time|Delayed_Arrivals|Average_Delay_Per_Delayed_Arrival|duplicate_count|
+----+------------+-----------------+---------------+-----------------------+---------------------+--------------------+------------------+----------------+-------------------------+-----------------------+----

In [6]:
from pyspark.sql.functions import col

numeric_columns = [
    "Actual_departures",
    "Actual_Arrivals",
    "Departure_Cancellations",
    "Arrival_Cancellations"
]

for c in numeric_columns:
    print(c,
          asqp_df.filter(col(c) < 0).count())

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 8, Finished, Available, Finished, False)

Actual_departures 0
Actual_Arrivals 0
Departure_Cancellations 0
Arrival_Cancellations 0


In [7]:
display(asqp_df.limit(5))

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c440cf9-f646-43d7-929a-7f8d9c69caae)

In [8]:
from pyspark.sql.functions import col

asqp_df = (
    asqp_df
    .withColumnRenamed("Date", "operation_date")
    .withColumnRenamed("Airport_Code", "airport_code")
    .withColumnRenamed("Actual_departures", "actual_departures")
    .withColumnRenamed("Actual_Arrivals", "actual_arrivals")
    .withColumnRenamed("Departure_Cancellations", "departure_cancellations")
    .withColumnRenamed("Arrival_Cancellations", "arrival_cancellations")
    .withColumnRenamed("Departure_Diversions", "departure_diversions")
    .withColumnRenamed("Arrival_Diversions", "arrival_diversions")
    .withColumnRenamed("On-Time_Arrivals", "on_time_arrivals")
    .withColumnRenamed("% On-Time_Gate_Departures", "on_time_gate_departure_pct")
    .withColumnRenamed("% On-Time_Gate_Arrivals", "on_time_gate_arrival_pct")
    .withColumnRenamed("Average_Gate_Departures_Delay", "avg_gate_departure_delay")
    .withColumnRenamed("Average__Gate_Arrival_delay", "avg_gate_arrival_delay")
    .withColumnRenamed("Average_Block_Delay", "avg_block_delay")
    .withColumnRenamed("Average_Taxi_out_time", "avg_taxi_out_time")
    .withColumnRenamed("Average_Taxi_in_Time", "avg_taxi_in_time")
    .withColumnRenamed("Delayed_Arrivals", "delayed_arrivals")
    .withColumnRenamed("Average_Delay_Per_Delayed_Arrival", "avg_delay_per_delayed_arrival")
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 10, Finished, Available, Finished, False)

In [9]:
print(asqp_df.columns)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 11, Finished, Available, Finished, False)

['operation_date', 'airport_code', 'actual_departures', 'actual_arrivals', 'departure_cancellations', 'arrival_cancellations', 'departure_diversions', 'arrival_diversions', 'on_time_arrivals', 'on_time_gate_departure_pct', 'on_time_gate_arrival_pct', 'avg_gate_departure_delay', 'avg_gate_arrival_delay', 'avg_block_delay', 'avg_taxi_out_time', 'avg_taxi_in_time', 'delayed_arrivals', 'avg_delay_per_delayed_arrival']


In [10]:
dim_airport = spark.read.table("dim_airport")

display(dim_airport.limit(5))

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fef630af-c149-4a57-9390-eb587d7180a7)

In [11]:
dim_airport_lookup = (
    dim_airport
    .select(
        "airport_key",
        col("iata_code").alias("airport_code")
    )
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 13, Finished, Available, Finished, False)

In [12]:
from pyspark.sql.functions import col

asqp_df = (
    asqp_df.alias("a")
    .join(
        dim_airport_lookup.alias("d"),
        col("a.airport_code") == col("d.airport_code"),
        "left"
    )
    .drop(col("d.airport_code"))
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 14, Finished, Available, Finished, False)

In [13]:
print("Rows :", asqp_df.count())

print(
    "Null Airport Keys :",
    asqp_df.filter(col("airport_key").isNull()).count()
)

display(
    asqp_df.select(
        "airport_code",
        "airport_key"
    ).limit(10)
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 15, Finished, Available, Finished, False)

Rows : 10590
Null Airport Keys : 0


SynapseWidget(Synapse.DataFrame, 746c8757-199f-425b-8434-4fafde483cba)

In [14]:
dim_date = spark.read.table("dim_date")

date_lookup = (
    dim_date
    .select(
        "flight_date",
        "date_key"
    )
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 16, Finished, Available, Finished, False)

In [15]:
from pyspark.sql.functions import col

asqp_df = (
    asqp_df.alias("a")
    .join(
        date_lookup.alias("d"),
        col("a.operation_date") == col("d.flight_date"),
        "left"
    )
    .drop("flight_date")
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 17, Finished, Available, Finished, False)

In [16]:
print("Rows :", asqp_df.count())

print(
    "Null Date Keys :",
    asqp_df.filter(col("date_key").isNull()).count()
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 18, Finished, Available, Finished, False)

Rows : 10590
Null Date Keys : 0


In [17]:
display(asqp_df.limit(5))

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee33d2ed-a5af-4109-861c-e2e4f1ffd7a3)

In [18]:
from pyspark.sql.functions import monotonically_increasing_id

fact_airport_operations = (
    asqp_df.select(
        "date_key",
        "airport_key",

        "actual_departures",
        "actual_arrivals",

        "departure_cancellations",
        "arrival_cancellations",

        "departure_diversions",
        "arrival_diversions",

        "on_time_gate_departure_pct",
        "on_time_gate_arrival_pct",

        "avg_gate_departure_delay",
        "avg_gate_arrival_delay",

        "avg_block_delay",

        "avg_taxi_out_time",
        "avg_taxi_in_time",

        "delayed_arrivals",
        "avg_delay_per_delayed_arrival"
    )
    .withColumn(
        "airport_operations_key",
        monotonically_increasing_id() + 1
    )
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 20, Finished, Available, Finished, False)

In [19]:
fact_airport_operations = fact_airport_operations.select(
    "airport_operations_key",
    "date_key",
    "airport_key",

    "actual_departures",
    "actual_arrivals",

    "departure_cancellations",
    "arrival_cancellations",

    "departure_diversions",
    "arrival_diversions",

    "on_time_gate_departure_pct",
    "on_time_gate_arrival_pct",

    "avg_gate_departure_delay",
    "avg_gate_arrival_delay",

    "avg_block_delay",

    "avg_taxi_out_time",
    "avg_taxi_in_time",

    "delayed_arrivals",
    "avg_delay_per_delayed_arrival"
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 21, Finished, Available, Finished, False)

In [20]:
print("Rows :", fact_airport_operations.count())

fact_airport_operations.printSchema()

display(fact_airport_operations.limit(10))

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 22, Finished, Available, Finished, False)

Rows : 10590
root
 |-- airport_operations_key: long (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- airport_key: integer (nullable = true)
 |-- actual_departures: long (nullable = true)
 |-- actual_arrivals: long (nullable = true)
 |-- departure_cancellations: long (nullable = true)
 |-- arrival_cancellations: long (nullable = true)
 |-- departure_diversions: long (nullable = true)
 |-- arrival_diversions: long (nullable = true)
 |-- on_time_gate_departure_pct: decimal(34,6) (nullable = true)
 |-- on_time_gate_arrival_pct: decimal(34,6) (nullable = true)
 |-- avg_gate_departure_delay: long (nullable = true)
 |-- avg_gate_arrival_delay: long (nullable = true)
 |-- avg_block_delay: long (nullable = true)
 |-- avg_taxi_out_time: long (nullable = true)
 |-- avg_taxi_in_time: long (nullable = true)
 |-- delayed_arrivals: long (nullable = true)
 |-- avg_delay_per_delayed_arrival: long (nullable = true)



SynapseWidget(Synapse.DataFrame, d5dedb92-b522-4eb5-8a11-9d4647511463)

In [21]:
print(
    fact_airport_operations.filter(col("airport_key").isNull()).count()
)

print(
    fact_airport_operations.filter(col("date_key").isNull()).count()
)

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 23, Finished, Available, Finished, False)

0
0


In [22]:
fact_airport_operations.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("fact_airport_operations")

StatementMeta(, 486a2a43-97cd-4f9b-818c-b4e0874511a1, 24, Finished, Available, Finished, False)